In [ ]:
# Cell 1 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 2 — Clone repo on first run, git pull on restarts (idempotent)
import os
from google.colab import userdata

token = userdata.get('GITHUB_TOKEN')
repo_dir = '/content/ebpf-fuzzing-thesis'

if not os.path.exists(repo_dir):
    ret = os.system(f'git clone https://{token}@github.com/Strhata/ebpf-fuzzing-thesis.git {repo_dir}')
    assert ret == 0, 'git clone failed'
else:
    ret = os.system(f'git -C {repo_dir} pull')
    assert ret == 0, 'git pull failed'

os.chdir(repo_dir)
print(f'Working directory: {os.getcwd()}')

In [ ]:
# Cell 3 — Install training-side dependencies (~10 min on a fresh runtime)
import subprocess
subprocess.run(
    ['pip', 'install', '-q', '-r', 'ml/requirements_colab.txt'],
    check=True,
)

In [ ]:
# Cell 3b — Merge SFT-v2 adapter into the base (run ONCE; idempotent).
# RL loads a MERGED fp16 model — a LoRA adapter on a 4-bit base collapses under GRPO
# (benchmark: merged_bnb4bit -> 0% valid). Output -> MERGED_OUT, which Cell 4 MODEL points at.
import os, glob, torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

BASE_MODEL = 'Qwen/Qwen2.5-Coder-1.5B'
ADAPTER    = '/content/drive/MyDrive/sft-1epoch-v2/sft_adapter'   # the SFT-v2 LoRA adapter
MERGED_OUT = '/content/drive/MyDrive/models/sft_v2_merged'        # == Cell 4 MODEL default

if os.path.isdir(MERGED_OUT) and glob.glob(f'{MERGED_OUT}/*.safetensors'):
    print(f'[skip] merged model already at {MERGED_OUT}')
else:
    assert glob.glob(f'{ADAPTER}/adapter_*'), f'no adapter found at {ADAPTER} — check the path'
    print('[*] loading base in bf16 ...')
    base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.bfloat16)
    print('[*] applying + merging adapter ...')
    merged = PeftModel.from_pretrained(base, ADAPTER).merge_and_unload()
    os.makedirs(MERGED_OUT, exist_ok=True)
    merged.save_pretrained(MERGED_OUT, safe_serialization=True)
    AutoTokenizer.from_pretrained(BASE_MODEL).save_pretrained(MERGED_OUT)   # RL needs the tokenizer
    print(f'[done] merged fp16 saved to {MERGED_OUT}')


In [ ]:
# Cell 4 — Config (the only cell you may need to edit)
# RE-RUN THIS CELL after any edit, then run Cell 5 — Colab won't pick up changes otherwise.
#
# MODEL: the MERGED fp16 SFT-v2 model (rl_grpo loads it bf16 — never 4-bit a merged fine-tune).
#   checkpoints/ is gitignored, so stage the merged weights to Drive first and point MODEL there.
#
# REWARD WEIGHTS ARE NOT SET HERE. The reward runs on your LOCAL WSL box (the reward server,
# next to the KCOV VM). Tune it there when you launch uvicorn:
#   RL_W_VALID=1.0  RL_W_NOVELTY=1.0  RL_W_REJECT_MAX=0.3            (phase A, smoke)
#   ...plus RL_W_GLOBAL=2.0  -> phase B: decayed-global novelty (the anti-clustering run)
# Colab only controls the training knobs below.
MODEL     = '/content/drive/MyDrive/models/sft_v2_merged'
G         = 16      # GRPO group size. Target 16 (~1.3 valid/group at ~7% valid). If it OOMs
                    #   the 40GB A100: drop MAX_LEN first (length!=diversity), then G->12, then 8.
MAX_LEN   = 512     # completion tokens. 512 justified: SFT-v2 512->1024 raised insns 30->58 but
                    #   unique PCs only +12% — length is not the diversity lever.
MAX_STEPS = 0       # 0 = unlimited (this IS the real phase-B run). Use 20 only for a smoke re-test.
RUN_NAME    = 'grpo-rlv2-phaseB-global'   # phase B (server launched with RL_W_GLOBAL=2.0)
OUTPUT_DIR  = f'/content/drive/MyDrive/{RUN_NAME}'
REWARD_URL  = userdata.get('REWARD_SERVER_URL')

print(f'MODEL={MODEL}')
print(f'G={G}  MAX_LEN={MAX_LEN}  MAX_STEPS={MAX_STEPS or "unlimited"}  RUN_NAME={RUN_NAME}')
print(f'OUTPUT_DIR={OUTPUT_DIR}')
print(f'REWARD_URL={REWARD_URL}')
print('\nPHASE-B WATCH (W&B) — the experiment:')
print('  novelty/global_frontier  climbs = clustering breaking | plateaus = clustering wins')
print('  novelty/global_mean      starts ~1.0, declines as the frontier fills (archive working)')
print('  valid_rate ~0.25 | reward/std > 0 | kl bounded | no NaN')


In [ ]:
# Cell 5 — Launch training (safe to re-run: --resume handles fresh start and continuation)
import os, subprocess
from google.colab import userdata

os.environ['WANDB_API_KEY']  = userdata.get('WANDB_API_KEY')
os.environ['REWARD_API_KEY'] = userdata.get('REWARD_API_KEY')

max_steps_flag = f' --max-steps {MAX_STEPS}' if MAX_STEPS and MAX_STEPS > 0 else ''

cmd = (
    f'python -u ml/rl_grpo.py'
    f' --resume'
    f' --model {MODEL}'
    f' --remote-reward-url {REWARD_URL}'
    f' --run-name {RUN_NAME}'
    f' --num-generations {G}'
    f' --max-completion-length {MAX_LEN}'
    f' --output-dir {OUTPUT_DIR}'
    f'{max_steps_flag}'
)
print('LAUNCH:', cmd, '\n')   # check this shows your G and max-steps before it runs
# Stream the child's pipe through Python print so Colab actually displays it (stdout+stderr merged).
# os.system / subprocess.run write to the raw OS fd, which Colab does NOT capture -> silent.
proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
proc.wait()
print(f'\n[exit {proc.returncode}]')
if proc.returncode != 0:
    print('[!] training exited non-zero — the traceback is just above')